# DLinear and PatchTST Split-Horizon Forecasting

This hybrid model uses DLinear for short-horizon forecasting and PatchTST for longer-horizon forecasting. The two 6-step outputs are concatenated into one 12-step forecast.

DLinear predicts forecast steps 1-6. PatchTST predicts forecast steps 7-12. Both branches use the same ETTh1 dataset, preprocessing, scaler, input length, batch size, and train/validation/test split.

## 1. Project Setup

This cell keeps the notebook runnable from inside `notebooks/` by moving to the repo root and adding `src/` to Python's import path.


In [ ]:
import os 
import sys
from pathlib import Path
#root contains the path to the basicts
ROOT = Path(r"C:\Users\luwil\OneDrive\Documents\Code\BasicTS")
#move python working folder
os.chdir(ROOT)
# the path to src is src_path
src_path = ROOT / "src"
#if src_path is not in the system path, add it to the system path
#system path is added to python search. 
# This allows us to import modules from the src 
# folder without having to specify the full path.
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

## 2. Imports and Shared Settings

Both models use the same dataset, scaler, preprocessing, `input_len`, train/val/test split, and batch format. The shared dataset keeps the full 12-step target window, while each split-horizon model trains on its own 6-step slice.


In [ ]:
import json
from datetime import datetime
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from basicts.configs import BasicTSForecastingConfig
from basicts.launcher import BasicTSLauncher
from basicts.models.DLinear import DLinear, DLinearConfig
from basicts.models.PatchTST import PatchTSTConfig, PatchTSTForForecasting
from basicts.runners.builder import Builder
from basicts.runners.taskflow import BasicTSForecastingTaskFlow
from basicts.scaler import ZScoreScaler
from basicts.utils import BasicTSMode
DATASET_NAME = "ETTh1"
#most papers use 96 input length
INPUT_LEN = 96
#most papers use 96, 192, 336, and 720 input length
FULL_OUTPUT_LEN = 12

SPLIT_OUTPUT_LEN = 6
#etthl has 7 variables
NUM_FEATURES = 7
#Batch sizes like 16, 32, and 64 are normal. 32 is a safe default.
BATCH_SIZE = 32
# common for testing
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3
# Fresh namespace for this notebook so BasicTS does not auto-resume from old/corrupt checkpoints.
RUN_TAG = "mixed_dlinear_patchtst"
# this is the shared settings dictionary that both models use
SHARED_CONFIG = {
    "dataset_name": DATASET_NAME,
    "input_len": INPUT_LEN,
    "dataset_params": {
        "input_len": INPUT_LEN,
        "output_len": FULL_OUTPUT_LEN,
        "use_timestamps": False,
        "memmap": False,
    },
    "use_timestamps": False,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "scaler": ZScoreScaler,
    "norm_each_channel": True,
    "rescale": False,
    "metrics": ["MAE", "MSE"],
    "optimizer_params": {"lr": LEARNING_RATE, "weight_decay": 5e-4},
    "gpus": None,
    "train_data_num_workers": 0,
    "val_data_num_workers": 0,
    "test_data_num_workers": 0,
    "save_results": True,
}

# The standalone 12-step models only need BasicTS test_metrics.json.
# Metrics-only evaluation avoids Windows memmap file-lock issues.
FULL_12_STEP_CONFIG = dict(SHARED_CONFIG)
FULL_12_STEP_CONFIG["save_results"] = False


def fresh_checkpoint_dir(model_folder, run_name):
    # Each training launch gets a unique parent folder, so BasicTS cannot resume a stale checkpoint.
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return str(Path("checkpoints") / RUN_TAG / model_folder / run_name / stamp)

## 3. Shared Shape Check Helper

This helper builds the BasicTS dataset and scaler, runs the same forecasting preprocessing that training uses, and then sends one batch through the selected model.


In [ ]:
# for the split forecasitting model, we need to create a custom taskflow that slices the targets and target masks to the desired output length      
#start with the forecasting taskflwo and mofify it 
# BasicTSForecastingTaskFlow is the default data-prep worker.
# It prepares each forecasting batch before the model uses it.
# start with the deault taskflow and then add a change 
class SplitHorizonForecastingTaskFlow(BasicTSForecastingTaskFlow):
    #adding a setting called targest slide whchi is a variable that is used to slice the targets
    # tells the taskflow which targets to keep
    # need self because we need acresss to this specific object
    #creates a variables incide the class object 
    def __init__(self, target_slice):
        self.target_slice = target_slice
    # preprocess 
    def preprocess(self, runner, data):
        #normal work
        data = super().preprocess(runner, data)
        #cuts target values
        data["targets"] = data["targets"][:, self.target_slice, :]
        # tells basicts which target values are valid
        data["targets_mask"] = data["targets_mask"][:, self.target_slice, :]
        return data


def _float_batch(batch):
    return {
        key: value.float() if isinstance(value, torch.Tensor) and value.is_floating_point() else value
        for key, value in batch.items()
    }

# checks to see if the inputs are the right shape the targest are sliced and the prediction matches targer
def preview_shapes(cfg, model_name):
    #build the dataset using basicts
    train_dataset = Builder._build_dataset(cfg, BasicTSMode.TRAIN)
    # puts the dataset into batches gets ready for batches
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=False)
    #This grabs the first batch.
    raw_batch = _float_batch(next(iter(train_loader)))
    #creates the scaler and fits it to the training data  
    scaler = Builder._build_scaler(cfg)
    scaler.fit(train_dataset.data)
    # fake runner so the pasicts 
    class PreviewRunner:
        pass
    # Create a fake runner that has cfg and scaler, because taskflow.preprocess expects a runner object.
    runner = PreviewRunner() 
    runner.cfg = cfg
    runner.scaler = scaler
    #prepares the batch normally
    processed_batch = cfg.taskflow.preprocess(runner, dict(raw_batch))
    #build the model from the config and switch it to evaluation mode
    model = cfg.model(cfg.model_config)
    model.eval()
    # o not track gradient as they are only needed for trainig 
    with torch.no_grad():
        #sends processed inputs into the model
        prediction = model(processed_batch["inputs"])
        # then the model outputs future values
        # did the model return a dictionary
        if isinstance(prediction, dict):
            # if it did this extracts onlt the prediction tensor
            prediction = prediction["prediction"]
    #print the shapres to check that the data and model match before the training
    print(f"{model_name} raw inputs shape:       ", tuple(raw_batch["inputs"].shape))
    print(f"{model_name} raw target shape:       ", tuple(raw_batch["targets"].shape))
    print(f"{model_name} processed inputs shape: ", tuple(processed_batch["inputs"].shape))
    print(f"{model_name} target shape:           ", tuple(processed_batch["targets"].shape))
    print(f"{model_name} prediction shape:       ", tuple(prediction.shape))
    #checks
    assert tuple(raw_batch["targets"].shape) == (cfg.batch_size, FULL_OUTPUT_LEN, NUM_FEATURES)
    assert tuple(processed_batch["inputs"].shape) == (cfg.batch_size, INPUT_LEN, NUM_FEATURES)
    assert tuple(processed_batch["targets"].shape) == (cfg.batch_size, SPLIT_OUTPUT_LEN, NUM_FEATURES)
    assert tuple(prediction.shape) == (cfg.batch_size, SPLIT_OUTPUT_LEN, NUM_FEATURES)
    return processed_batch, prediction


## 4. DLinear Model

This notebook uses BasicTS's built-in `DLinear` model for forecast steps 1-6. `DLinear` receives `[batch_size, input_len, num_features]` and returns `[batch_size, 6, num_features]` for the split-horizon branch.

In [ ]:
# DLinear is imported from BasicTS in the imports cell:
# from basicts.models.DLinear import DLinear, DLinearConfig
# No custom model class is needed for this branch.

## 5. DLinear Config

This config reuses `BasicTSForecastingConfig` and only changes the model-specific pieces. The shared dataset/scaler/preprocessing settings come from `SHARED_CONFIG`.


In [ ]:
# tells BasicTS how to build and train DLinear
dlinear_model_config = DLinearConfig(
    input_len=INPUT_LEN,
    output_len=SPLIT_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    moving_avg=25,
    stride=1,
    individual=False,
)

# full BasicTS training config for split-horizon DLinear
dlinear_cfg = BasicTSForecastingConfig(
    model=DLinear,
    model_config=dlinear_model_config,
    taskflow=SplitHorizonForecastingTaskFlow(slice(0, SPLIT_OUTPUT_LEN)),
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/DLinear/{DATASET_NAME}_{INPUT_LEN}_steps_1_6",
    **SHARED_CONFIG,
)

# Standalone DLinear trained to forecast all 12 steps.
dlinear_full_model_config = DLinearConfig(
    input_len=INPUT_LEN,
    output_len=FULL_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    moving_avg=25,
    stride=1,
    individual=False,
)

dlinear_full_cfg = BasicTSForecastingConfig(
    model=DLinear,
    model_config=dlinear_full_model_config,
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/DLinear/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **FULL_12_STEP_CONFIG,
)

dlinear_cfg, dlinear_full_cfg

## 6. DLinear Shape Test

Run this before training. The model prediction and processed target lines must be `(batch_size, 6, num_features)`, while the raw target line remains `(batch_size, 12, num_features)`.


In [ ]:
#checker
dlinear_batch, dlinear_prediction = preview_shapes(dlinear_cfg, "DLinear")


## 7. Train the DLinear

This is the first training run. Leave `RUN_DLinear_TRAINING` as `False` while editing or shape-checking, then switch it to `True` when you are ready to train.


In [ ]:
#training
RUN_DLinear_TRAINING = False
RUN_DLinear_12_STEP_TRAINING = False

if RUN_DLinear_TRAINING:
    dlinear_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "DLinear",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_6",
    )
    print("Training split DLinear in:", dlinear_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(dlinear_cfg)
else:
    print("DLinear split-horizon training skipped. Set RUN_DLinear_TRAINING = True to train.")

if RUN_DLinear_12_STEP_TRAINING:
    dlinear_full_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "DLinear",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    )
    print("Training 12-step DLinear in:", dlinear_full_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(dlinear_full_cfg)
else:
    print("DLinear 12-step training skipped. Set RUN_DLinear_12_STEP_TRAINING = True to train.")


## 8. PatchTST Model

PatchTST is responsible for forecast steps 7-12. It receives the same `[batch_size, input_len, num_features]` inputs as DLinear, but its taskflow slices the target window to the second half of the horizon.

In [ ]:
# PatchTST model config for forecast steps 7-12
patchtst_model_config = PatchTSTConfig(
    input_len=INPUT_LEN,
    output_len=SPLIT_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    patch_len=16,
    patch_stride=8,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    attn_dropout=0.1,
    fc_dropout=0.1,
    head_dropout=0.0,
    use_revin=True,
)

patchtst_cfg = BasicTSForecastingConfig(
    model=PatchTSTForForecasting,
    model_config=patchtst_model_config,
    taskflow=SplitHorizonForecastingTaskFlow(slice(SPLIT_OUTPUT_LEN, FULL_OUTPUT_LEN)),
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/PatchTSTForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_7_12",
    **SHARED_CONFIG,
)

# Standalone PatchTST trained to forecast all 12 steps.
patchtst_full_model_config = PatchTSTConfig(
    input_len=INPUT_LEN,
    output_len=FULL_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    patch_len=16,
    patch_stride=8,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    attn_dropout=0.1,
    fc_dropout=0.1,
    head_dropout=0.0,
    use_revin=True,
)

patchtst_full_cfg = BasicTSForecastingConfig(
    model=PatchTSTForForecasting,
    model_config=patchtst_full_model_config,
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/PatchTSTForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **FULL_12_STEP_CONFIG,
)

patchtst_cfg, patchtst_full_cfg

## 9. PatchTST Shape Test

Run this after the DLinear section works. It uses the same shared data pipeline and checks that the PatchTST predicts only its 6-step split-horizon target.


In [ ]:
#check the shapes
patchtst_batch, patchtst_prediction = preview_shapes(patchtst_cfg, "PatchTST")


## 10. Train PatchTST

Use the same dataset, scaler, preprocessing, and `INPUT_LEN = 96` as DLinear. The shared dataset still contains the full 12-step target window, but the PatchTST taskflow slices that target to steps 7-12.

In [ ]:
# train PatchTST
RUN_PATCHTST_TRAINING = False
RUN_PATCHTST_12_STEP_TRAINING = False

if RUN_PATCHTST_TRAINING:
    patchtst_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "PatchTSTForForecasting",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_7_12",
    )
    print("Training split PatchTST in:", patchtst_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(patchtst_cfg)
else:
    print("PatchTST split-horizon training skipped. Set RUN_PATCHTST_TRAINING = True after DLinear works.")

if RUN_PATCHTST_12_STEP_TRAINING:
    patchtst_full_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "PatchTSTForForecasting",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    )
    print("Training 12-step PatchTST in:", patchtst_full_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(patchtst_full_cfg)
else:
    print("PatchTST 12-step training skipped. Set RUN_PATCHTST_12_STEP_TRAINING = True to train.")

## 11. Hybrid Prediction: DLinear Steps 1-6, PatchTST Steps 7-12

This is a true split-horizon hybrid. The DLinear predicts only the first 6 forecast steps, the PatchTST predicts only the next 6 forecast steps from the same input window, and the final hybrid prediction is the time-axis concatenation of those two 6-step outputs.


In [ ]:
import numpy as np

#checks that the dlinear and patchtst both used the same batch
assert torch.equal(dlinear_batch["inputs"], patchtst_batch["inputs"])

# Split-horizon hybrid: DLinear predicts steps 1-6, PatchTST predicts steps 7-12.
dlinear_pred = dlinear_prediction
patchtst_pred = patchtst_prediction
#this combines the two step prediction into one 12 step hybrui prediction
hybrid_pred = torch.cat([dlinear_pred, patchtst_pred], dim=1)
#combines the targests together
hybrid_targets = torch.cat([dlinear_batch["targets"], patchtst_batch["targets"]], dim=1)
# checks 
print("DLinear prediction shape:", tuple(dlinear_pred.shape))
print("PatchTST prediction shape:", tuple(patchtst_pred.shape))
print("Hybrid prediction shape:", tuple(hybrid_pred.shape))
print("Target shape:", tuple(hybrid_targets.shape))

assert tuple(dlinear_pred.shape) == (BATCH_SIZE, SPLIT_OUTPUT_LEN, NUM_FEATURES)
assert tuple(patchtst_pred.shape) == (BATCH_SIZE, SPLIT_OUTPUT_LEN, NUM_FEATURES)
assert tuple(hybrid_pred.shape) == (BATCH_SIZE, FULL_OUTPUT_LEN, NUM_FEATURES)
assert tuple(hybrid_targets.shape) == (BATCH_SIZE, FULL_OUTPUT_LEN, NUM_FEATURES)

#find the last saved prediction
def latest_prediction_file(cfg):
    prediction_files = sorted(
        Path(cfg.ckpt_save_dir).rglob("test_results/prediction.npy"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not prediction_files:
        raise FileNotFoundError(
            f"No prediction.npy found under {cfg.ckpt_save_dir}. Train/evaluate this model with save_results=True first."
        )
    return prediction_files[0]

#load the prediction arrays
def load_basicts_array(path, shape):
    # BasicTS writes these files as raw memmaps, even though the file names end in .npy.
    array = np.memmap(path, dtype=np.float32, mode="r", shape=shape)
    return np.asarray(array)

#loads the largets
def load_basicts_prediction_and_targets(cfg, output_len):
    prediction_path = latest_prediction_file(cfg)
    targets_path = prediction_path.parent / "targets.npy"
    if not targets_path.exists():
        raise FileNotFoundError(f"No targets.npy found next to {prediction_path}")

    test_dataset = Builder._build_dataset(cfg, BasicTSMode.TEST)
    shape = (len(test_dataset), output_len, NUM_FEATURES)
    prediction = load_basicts_array(prediction_path, shape)
    targets = load_basicts_array(targets_path, shape)
    return prediction, targets


dlinear_test_pred, dlinear_test_targets = load_basicts_prediction_and_targets(dlinear_cfg, SPLIT_OUTPUT_LEN)
patchtst_test_pred, patchtst_test_targets = load_basicts_prediction_and_targets(patchtst_cfg, SPLIT_OUTPUT_LEN)
#bombines the test predictions
hybrid_test_pred = np.concatenate([dlinear_test_pred, patchtst_test_pred], axis=1)
hybrid_test_targets = np.concatenate([dlinear_test_targets, patchtst_test_targets], axis=1)

assert dlinear_test_pred.shape == dlinear_test_targets.shape
assert patchtst_test_pred.shape == patchtst_test_targets.shape
assert dlinear_test_pred.shape == patchtst_test_pred.shape
assert hybrid_test_pred.shape == hybrid_test_targets.shape
assert hybrid_test_pred.shape[1] == FULL_OUTPUT_LEN

print("DLinear prediction shape:", dlinear_test_pred.shape)
print("PatchTST prediction shape:", patchtst_test_pred.shape)
print("Hybrid prediction shape:", hybrid_test_pred.shape)
print("Hybrid target shape:", hybrid_test_targets.shape)

mixed_output_dir = Path("checkpoints") / RUN_TAG
mixed_output_dir.mkdir(parents=True, exist_ok=True)
hybrid_save_path = mixed_output_dir / "hybrid_split_horizon_dlinear_steps_1_6_patchtst_steps_7_12_ETTh1_96_12_prediction.npy"
np.save(hybrid_save_path, hybrid_test_pred)
print(f"Saved fixed split hybrid prediction: {hybrid_save_path}")

## 12. MAE/MSE Comparison

This cell computes MAE/MSE directly from the saved predictions and targets. The DLinear is compared only against target steps 1-6, the PatchTST only against target steps 7-12, and the hybrid against the full 12-step target.


In [ ]:
# Compute MAE/MSE metrics.
def compute_metrics(prediction, targets):
    return {
        "MAE": float(np.mean(np.abs(prediction - targets))),
        "MSE": float(np.mean((prediction - targets) ** 2)),
    }


def latest_metrics_file(cfg):
    metrics_files = sorted(
        Path(cfg.ckpt_save_dir).rglob("test_metrics.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not metrics_files:
        return None
    return metrics_files[0]


def load_test_metrics_if_available(cfg):
    metrics_path = latest_metrics_file(cfg)
    if metrics_path is None:
        return None
    with metrics_path.open("r", encoding="utf-8") as f:
        metrics = json.load(f)
    return metrics.get("overall", metrics)


comparison = {
    "DLinear steps 1-6": compute_metrics(dlinear_test_pred, dlinear_test_targets),
    "PatchTST steps 7-12": compute_metrics(patchtst_test_pred, patchtst_test_targets),
    "Hybrid DLinear + PatchTST steps 1-12": compute_metrics(hybrid_test_pred, hybrid_test_targets),
}

for model_name, cfg in [
    ("Full 12-step DLinear", dlinear_full_cfg),
    ("Full 12-step PatchTST", patchtst_full_cfg),
]:
    metrics = load_test_metrics_if_available(cfg)
    if metrics is not None:
        comparison[model_name] = metrics

print("DLinear target shape:", dlinear_test_targets.shape)
print("PatchTST target shape:", patchtst_test_targets.shape)
print("Hybrid target shape:", hybrid_test_targets.shape)

print(f"{'Model':<45} {'MAE':>12} {'MSE':>12}")
print("-" * 73)
for model_name, metrics in comparison.items():
    print(f"{model_name:<45} {metrics['MAE']:>12.6f} {metrics['MSE']:>12.6f}")

## 13. Efficiency Comparison

This cell compares model size and average batch prediction time. The DLinear timing is for its 6-step prediction, the PatchTST timing is for its 6-step prediction, and the hybrid timing is the sum of running both 6-step models once.


In [ ]:
import time


def count_trainable_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def time_model_prediction(model, batch, expected_output_len, repeats=50, warmup=5):
    device = next(model.parameters()).device
    inputs = batch["inputs"].to(device)
    model.eval()

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(inputs)

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(repeats):
            prediction = model(inputs)
    end = time.perf_counter()

    if isinstance(prediction, dict):
        prediction = prediction["prediction"]
    assert tuple(prediction.shape) == (inputs.size(0), expected_output_len, NUM_FEATURES)
    return (end - start) / repeats


dlinear_efficiency_model = dlinear_cfg.model(dlinear_cfg.model_config)
patchtst_efficiency_model = patchtst_cfg.model(patchtst_cfg.model_config)

dlinear_params = count_trainable_parameters(dlinear_efficiency_model)
patchtst_params = count_trainable_parameters(patchtst_efficiency_model)
hybrid_params = dlinear_params + patchtst_params

dlinear_time = time_model_prediction(dlinear_efficiency_model, dlinear_batch, SPLIT_OUTPUT_LEN)
patchtst_time = time_model_prediction(patchtst_efficiency_model, patchtst_batch, SPLIT_OUTPUT_LEN)
hybrid_time = dlinear_time + patchtst_time

print("DLinear parameters:", dlinear_params)
print("PatchTST parameters:", patchtst_params)
print("Hybrid total parameters:", hybrid_params)
print(f"DLinear 6-step avg prediction time: {dlinear_time:.6f} seconds")
print(f"PatchTST 6-step avg prediction time: {patchtst_time:.6f} seconds")
print(f"Hybrid total avg prediction time: {hybrid_time:.6f} seconds")

print(f"{'Model':<28} {'Params':>12} {'Avg batch sec':>15}")
print("-" * 58)
for model_name, params, avg_time in [
    ("DLinear", dlinear_params, dlinear_time),
    ("PatchTST", patchtst_params, patchtst_time),
    ("Hybrid total", hybrid_params, hybrid_time),
]:
    print(f"{model_name:<28} {params:>12,} {avg_time:>15.6f}")

## 14. Save Hybrid Outputs

Save the DLinear + PatchTST hybrid prediction array and metrics under `checkpoints/mixed_dlinear_patchtst/`.


In [ ]:
# Save hybrid predictions and metrics to checkpoints.
mixed_output_dir = Path("checkpoints") / RUN_TAG
mixed_output_dir.mkdir(parents=True, exist_ok=True)

hybrid_save_path = mixed_output_dir / "hybrid_dlinear_patchtst_ETTh1_96_12_prediction.npy"
np.save(hybrid_save_path, hybrid_test_pred)
print(f"Saved hybrid prediction: {hybrid_save_path}")

metrics_save_path = mixed_output_dir / "hybrid_dlinear_patchtst_metrics_ETTh1_96_12.json"
with metrics_save_path.open("w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=2)
print(f"Saved hybrid metrics: {metrics_save_path}")